# PD2 Intel Trader - Price Discovery 2
**Asymmetric Information Exploitation via Multi-User Intelligence**

Scrapes private estimates from user1-user20, intersects implied ranges to compute fair value, trades directionally.

Formula: `E_t = P_UB - X*(300-t)/50` → P_UB ∈ [E-(300-t)/50, E+(300-t)/50]

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import tkinter as tk
from tkinter import ttk
import time
import requests
import base64

In [27]:
# ============================================================
# CONFIGURATION
# ============================================================
BASE_URL     = 'http://141.211.79.101:10002'
API_PORT     = 10001
REMOTE_HOSTS = [BASE_URL, 'http://141.211.79.101:10001']
LOCAL_HOSTS  = [f'http://localhost:{API_PORT}', 'http://localhost:10002', 'http://localhost:9999']
HOST         = BASE_URL                      # Legacy alias used by older cells
ALT_HOST     = REMOTE_HOSTS[1]
LOCAL_HOST   = LOCAL_HOSTS[0]
API_KEY      = 'EOGIEYQN'
MY_USERNAME  = 'user4'
MY_PASSWORD  = 'user4'
TICKER       = 'UB'
TOTAL_TIME   = 300
POSITION_LIMIT = 10000
COMMISSION   = 0.02
MAX_ORDER_SIZE = 10000
PRIOR_LOW    = 40.0
PRIOR_HIGH   = 60.0

ALL_USERS = [f'user{i}' for i in range(1, 21)]

class ApiException(Exception):
    pass

In [28]:
class RITGateway:
    def __init__(self, host, username=None, password=None, api_key=None, label=None):
        self.host = host.rstrip('/')
        self.base_url = f'{self.host}/v1'
        self.session = requests.Session()
        self.username = username
        self.api_key = api_key
        self.label = label or self.host
        self.last_error = None
        self.auth_mode = 'api_key' if api_key else 'basic'

        if api_key:
            self.session.headers.update({'X-API-key': api_key})
        elif username is not None and password is not None:
            auth = base64.b64encode(f'{username}:{password}'.encode()).decode()
            self.session.headers.update({'Authorization': f'Basic {auth}'})
        else:
            raise ValueError('Provide either api_key or username/password')

    def _request(self, method, endpoint, params=None, timeout=None):
        url = f'{self.base_url}/{endpoint}'
        try:
            request_kwargs = {}
            if params is not None:
                request_kwargs['params'] = params
            if timeout is not None:
                request_kwargs['timeout'] = timeout

            if method == 'GET':
                resp = self.session.get(url, **request_kwargs)
            elif method == 'POST':
                resp = self.session.post(url, **request_kwargs)
            elif method == 'DELETE':
                resp = self.session.delete(url, **request_kwargs)
            else:
                raise ValueError(f'Unsupported method: {method}')

            if resp.status_code == 401:
                raise ApiException(f'Auth failed via {self.label}')
            if resp.status_code == 429:
                time.sleep(float(resp.headers.get('Retry-After', 0.5)))
                return self._request(method, endpoint, params=params, timeout=timeout)
            if not resp.ok:
                raise ApiException(f'{resp.status_code} {resp.text}')

            self.last_error = None
            return resp.json() if resp.text.strip() else None
        except requests.exceptions.Timeout as e:
            self.last_error = f'Timeout contacting {self.base_url}: {e}'
            return None
        except requests.exceptions.ConnectionError as e:
            self.last_error = f'ConnectionError contacting {self.base_url}: {e}'
            return None
        except Exception as e:
            self.last_error = f'{type(e).__name__}: {e}'
            return None

    def get_case(self, timeout=None):
        return self._request('GET', 'case', timeout=timeout)

    def get_securities(self, timeout=None):
        return self._request('GET', 'securities', timeout=timeout)

    def get_book(self, ticker, timeout=None):
        return self._request('GET', 'securities/book', params={'ticker': ticker}, timeout=timeout)

    def get_news(self, since=0, timeout=None):
        return self._request('GET', 'news', params={'since': since}, timeout=timeout)

    def cancel_all_orders(self, timeout=None):
        return self._request('POST', 'commands/cancel', params={'all': 1}, timeout=timeout)

    def get_security(self, ticker, timeout=None):
        secs = self.get_securities(timeout=timeout)
        if secs is None:
            return None
        for s in secs:
            if s.get('ticker') == ticker:
                return s
        return None

    def submit_order(self, ticker, order_type, quantity, action, price=None, timeout=None):
        params = {'ticker': ticker, 'type': order_type, 'quantity': int(quantity), 'action': action}
        if price is not None:
            params['price'] = round(float(price), 2)
        return self._request('POST', 'orders', params=params, timeout=timeout)


def build_main_gateway_candidates():
    candidates = []
    seen = set()

    def add_gateway(host, username=None, password=None, api_key=None, label=None):
        host = host.rstrip('/')
        key = (host, username or '', api_key or '', label or '')
        if key in seen:
            return
        seen.add(key)
        candidates.append(RITGateway(host=host, username=username, password=password, api_key=api_key, label=label))

    # 1. Remote basic auth first (same approach as LT2 - works without local RIT client)
    for host in REMOTE_HOSTS:
        add_gateway(host=host, username=MY_USERNAME, password=MY_PASSWORD, label=f'remote_basic:{MY_USERNAME}@{host}')

    # 2. Local with API key (fallback if RIT client is running locally)
    if API_KEY:
        for host in LOCAL_HOSTS:
            add_gateway(host=host, api_key=API_KEY, label=f'local_api_key:{host}')
        add_gateway(host=BASE_URL, api_key=API_KEY, label=f'remote_api_key:{BASE_URL}')

    # 3. Local basic auth
    for host in LOCAL_HOSTS:
        add_gateway(host=host, username=MY_USERNAME, password=MY_PASSWORD, label=f'local_basic:{MY_USERNAME}@{host}')

    return candidates


def select_gateway(candidates, timeout=4):
    diagnostics = []
    for gw in candidates:
        case_data = gw.get_case(timeout=timeout)
        if case_data is not None:
            return gw, diagnostics
        diagnostics.append((gw.label, gw.last_error or 'No response'))
    return None, diagnostics


def diagnose_rit_connection(timeout=4):
    print('[DIAG] Testing RIT connection modes...')
    for gw in build_main_gateway_candidates():
        case_data = gw.get_case(timeout=timeout)
        if case_data is not None:
            print(f"[DIAG] {gw.label}: OK tick={case_data.get('tick')} status={case_data.get('status')!r}")
        else:
            print(f"[DIAG] {gw.label}: ERROR {gw.last_error or 'No response'}")

In [29]:
class IntelAggregator:
    """
    Scrapes user1-user20 for private news/estimates.
    Intersects implied ranges to compute tightest fair value.
    Auto-detects which users are actually responding; not all 20 need to be active.
    """

    def __init__(self, users=None, main_gateway=None, remote_hosts=None):
        self.users = users or ALL_USERS
        self.main_gateway = main_gateway
        self.remote_hosts = remote_hosts or REMOTE_HOSTS
        self.gateway_candidates = {u: self._build_candidates(u) for u in self.users}
        self.selected_gateways = {}
        self.all_estimates = []
        self.estimates_by_user = {u: [] for u in self.users}
        self.last_news_id = {u: 0 for u in self.users}
        self.fail_count = {u: 0 for u in self.users}
        self.dead_users = set()
        self.range_low = PRIOR_LOW
        self.range_high = PRIOR_HIGH
        self.fair_value = (PRIOR_LOW + PRIOR_HIGH) / 2
        self.confidence = 0.0

    def _build_candidates(self, user):
        candidates = []
        seen = set()

        def add_gateway(gw):
            key = (gw.host, gw.auth_mode, gw.username or '', gw.api_key or '')
            if key in seen:
                return
            seen.add(key)
            candidates.append(gw)

        if user == MY_USERNAME and self.main_gateway is not None:
            add_gateway(self.main_gateway)

        for host in self.remote_hosts:
            add_gateway(RITGateway(host=host, username=user, password=user, label=f'remote_basic:{user}@{host}'))

        return candidates

    def _probe_user(self, user, timeout=1):
        for gateway in self.gateway_candidates.get(user, []):
            case_data = gateway.get_case(timeout=timeout)
            if case_data is not None:
                self.selected_gateways[user] = gateway
                return gateway
        self.selected_gateways.pop(user, None)
        return None

    def probe_users(self, timeout=1):
        """Check which users are actually reachable once the case is live."""
        print(f'[INTEL] Probing {len(self.users)} candidate users (timeout={timeout}s)...')
        active_users = []
        for user in self.users:
            if user in self.dead_users:
                continue
            if self._probe_user(user, timeout=timeout) is None:
                self.dead_users.add(user)
            else:
                active_users.append(user)

        sample = ', '.join(active_users[:8])
        if len(active_users) > 8:
            sample += ', ...'
        if sample:
            print(f'[INTEL] {len(active_users)}/{len(self.users)} users responding: {sample}')
        else:
            print(f'[INTEL] {len(active_users)}/{len(self.users)} users responding')
        return len(active_users)

    def parse_estimates(self, news_item):
        if isinstance(news_item, dict):
            body = str(news_item.get('body', '')) + ' ' + str(news_item.get('headline', ''))
            tick = news_item.get('tick', None)
        else:
            body, tick = str(news_item), None
        vals = [float(m) for m in re.findall(r'\$?([\d]+\.[\d]{2})', body)
                if 30.0 <= float(m) <= 70.0]
        return vals, tick

    def scrape_all_users(self, current_tick=None):
        new_count = 0
        for user in self.users:
            if user in self.dead_users:
                continue

            gateway = self.selected_gateways.get(user)
            if gateway is None:
                gateway = self._probe_user(user, timeout=1)
                if gateway is None:
                    self.fail_count[user] += 1
                    if self.fail_count[user] >= 2:
                        self.dead_users.add(user)
                    continue

            try:
                news = gateway.get_news(since=self.last_news_id[user], timeout=1)
                if news is None:
                    self.fail_count[user] += 1
                    self.selected_gateways.pop(user, None)
                    if self.fail_count[user] >= 3:
                        self.dead_users.add(user)
                        print(f'[INTEL] {user} -> dead ({gateway.last_error or "no response"})')
                    continue

                self.fail_count[user] = 0
                for item in (news if isinstance(news, list) else []):
                    nid = item.get('news_id', item.get('id', 0))
                    if nid > self.last_news_id[user]:
                        self.last_news_id[user] = nid
                    estimates, tick = self.parse_estimates(item)
                    news_tick = tick if tick is not None else current_tick
                    for est in estimates:
                        if not any(e['user'] == user and e['estimate'] == est and e['tick'] == news_tick for e in self.all_estimates):
                            entry = {
                                'user': user,
                                'estimate': est,
                                'tick': news_tick,
                                'news_id': nid,
                                'raw': item.get('body', item.get('headline', '')),
                            }
                            self.all_estimates.append(entry)
                            self.estimates_by_user[user].append(entry)
                            new_count += 1
            except Exception as e:
                self.fail_count[user] += 1
                self.selected_gateways.pop(user, None)
                if self.fail_count[user] >= 3:
                    self.dead_users.add(user)
                    print(f'[INTEL] {user} -> dead ({type(e).__name__}: {e})')

        if new_count > 0:
            self._recompute_range(current_tick)
        return new_count

    def _recompute_range(self, current_tick=None):
        low, high = PRIOR_LOW, PRIOR_HIGH
        for e in self.all_estimates:
            if e['tick'] is None:
                continue
            eb = (300 - e['tick']) / 50.0
            low = max(low, e['estimate'] - eb)
            high = min(high, e['estimate'] + eb)
        low = max(low, PRIOR_LOW)
        high = min(high, PRIOR_HIGH)
        if low <= high:
            self.range_low = round(low, 2)
            self.range_high = round(high, 2)
            self.fair_value = round((low + high) / 2, 2)
            self.confidence = max(0.0, min(1.0, 1.0 - (high - low) / (PRIOR_HIGH - PRIOR_LOW)))

    def get_summary(self):
        return {
            'range_low': self.range_low,
            'range_high': self.range_high,
            'fair_value': self.fair_value,
            'confidence': self.confidence,
            'range_width': round(self.range_high - self.range_low, 2),
            'total_estimates': len(self.all_estimates),
            'users_with_data': sum(1 for u in self.users if self.estimates_by_user[u]),
            'active_users': len(self.users) - len(self.dead_users),
        }

In [30]:
class PD2Trader:
    def __init__(self, intel,
                 position_limit=POSITION_LIMIT, commission=COMMISSION,
                 passive_threshold=0.3, active_threshold=0.5, aggressive_threshold=0.8,
                 base_size=500, max_aggressive_size=5000,
                 endgame_start=260, hard_flatten_start=280, final_flatten_start=292):
        self.intel = intel
        self.position_limit = position_limit
        self.commission = commission
        self.passive_threshold = passive_threshold
        self.active_threshold = active_threshold
        self.aggressive_threshold = aggressive_threshold
        self.base_size = base_size
        self.max_aggressive_size = max_aggressive_size
        self.endgame_start = endgame_start
        self.hard_flatten_start = hard_flatten_start
        self.final_flatten_start = final_flatten_start

    def step(self, best_bid, best_ask, inventory, tick, fair_value, confidence, range_low, range_high):
        if tick >= self.final_flatten_start and abs(inventory) > 0:
            return self._flatten(inventory, best_bid, best_ask, 'ENDGAME FLATTEN')
        if tick >= self.hard_flatten_start and abs(inventory) > 500:
            return self._flatten(inventory, best_bid, best_ask, 'HARD FLATTEN')
        if tick >= self.endgame_start and abs(inventory) > 2000:
            return self._flatten(inventory, best_bid, best_ask, 'ENDGAME REDUCE')

        edge_buy  = fair_value - best_ask - self.commission
        edge_sell = best_bid - fair_value - self.commission
        rw = range_high - range_low

        if confidence >= self.aggressive_threshold:
            return self._aggressive(best_bid, best_ask, inventory, fair_value, edge_buy, edge_sell, rw, confidence)
        elif confidence >= self.active_threshold:
            return self._active(best_bid, best_ask, inventory, fair_value, edge_buy, edge_sell, rw, confidence)
        else:
            return self._passive(best_bid, best_ask, inventory, fair_value, confidence)

    def _aggressive(self, best_bid, best_ask, inventory, fv, edge_buy, edge_sell, rw, conf):
        orders = []
        size_mult = min(3.0, 1.0 + (conf - self.aggressive_threshold) * 10)
        size = int(min(self.max_aggressive_size, self.base_size * size_mult) / 100) * 100
        if edge_buy > 0 and inventory < self.position_limit * 0.8:
            qty = max(100, int(min(size, (self.position_limit - inventory) * 0.5) / 100) * 100)
            orders.append({'action': 'BUY', 'type': 'LIMIT',
                           'price': round(min(best_ask, fv - self.commission), 2), 'quantity': qty})
            if rw < 2.0:
                orders.append({'action': 'BUY', 'type': 'LIMIT',
                               'price': round(max(PRIOR_LOW, fv - rw - 0.10), 2),
                               'quantity': min(2000, int((self.position_limit - inventory) * 0.3))})
        elif edge_sell > 0 and inventory > -self.position_limit * 0.8:
            qty = max(100, int(min(size, (self.position_limit + inventory) * 0.5) / 100) * 100)
            orders.append({'action': 'SELL', 'type': 'LIMIT',
                           'price': round(max(best_bid, fv + self.commission), 2), 'quantity': qty})
            if rw < 2.0:
                orders.append({'action': 'SELL', 'type': 'LIMIT',
                               'price': round(min(PRIOR_HIGH, fv + rw + 0.10), 2),
                               'quantity': min(2000, int((self.position_limit + inventory) * 0.3))})
        else:
            half = max(0.02, rw * 0.15)
            orders.append({'action': 'BUY',  'type': 'LIMIT', 'price': round(fv - half, 2),
                           'quantity': min(1000, max(100, int((self.position_limit - inventory) * 0.2)))})
            orders.append({'action': 'SELL', 'type': 'LIMIT', 'price': round(fv + half, 2),
                           'quantity': min(1000, max(100, int((self.position_limit + inventory) * 0.2)))})
        return {'mode': 'AGGRESSIVE', 'orders': orders, 'confidence': conf}

    def _active(self, best_bid, best_ask, inventory, fv, edge_buy, edge_sell, rw, conf):
        orders = []
        size = max(100, int(self.base_size * conf / 100) * 100)
        if edge_buy > 0.05 and inventory < self.position_limit * 0.5:
            qty = max(100, int(min(size, (self.position_limit - inventory) * 0.3) / 100) * 100)
            orders.append({'action': 'BUY', 'type': 'LIMIT',
                           'price': round(min(best_ask - 0.01, fv - 0.05), 2), 'quantity': qty})
        elif edge_sell > 0.05 and inventory > -self.position_limit * 0.5:
            qty = max(100, int(min(size, (self.position_limit + inventory) * 0.3) / 100) * 100)
            orders.append({'action': 'SELL', 'type': 'LIMIT',
                           'price': round(max(best_bid + 0.01, fv + 0.05), 2), 'quantity': qty})
        else:
            half = max(0.05, rw * 0.2)
            if inventory < self.position_limit * 0.5:
                orders.append({'action': 'BUY', 'type': 'LIMIT', 'price': round(fv - half, 2),
                               'quantity': min(500, max(100, int((self.position_limit - inventory) * 0.15)))})
            if inventory > -self.position_limit * 0.5:
                orders.append({'action': 'SELL', 'type': 'LIMIT', 'price': round(fv + half, 2),
                               'quantity': min(500, max(100, int((self.position_limit + inventory) * 0.15)))})
        return {'mode': 'ACTIVE', 'orders': orders, 'confidence': conf}

    def _passive(self, best_bid, best_ask, inventory, fv, conf):
        orders = []
        mid  = (best_bid + best_ask) / 2
        half = max((best_ask - best_bid) * 0.6, 0.15)
        if abs(inventory) < 2000:
            if inventory < self.position_limit * 0.3:
                orders.append({'action': 'BUY',  'type': 'LIMIT',
                               'price': round(mid - half, 2), 'quantity': 200})
            if inventory > -self.position_limit * 0.3:
                orders.append({'action': 'SELL', 'type': 'LIMIT',
                               'price': round(mid + half, 2), 'quantity': 200})
        return {'mode': 'PASSIVE', 'orders': orders, 'confidence': conf}

    def _flatten(self, inventory, best_bid, best_ask, reason):
        orders = []
        qty = max(100, int(min(abs(inventory), MAX_ORDER_SIZE) / 100) * 100)
        if inventory > 0:
            orders.append({'action': 'SELL', 'type': 'MARKET', 'quantity': qty})
        elif inventory < 0:
            orders.append({'action': 'BUY',  'type': 'MARKET', 'quantity': qty})
        return {'mode': reason, 'orders': orders, 'confidence': 0}

In [31]:
class PD2Dashboard:
    def __init__(self):
        if tk._default_root is not None:
            try: tk._default_root.destroy()
            except: pass

        self.root = tk.Tk()
        self.root.title('PD2 Intel Trader')
        self.root.geometry('900x750')
        self.root.configure(bg='#0a0a0a')
        self.gateway = None
        self.stop_requested = False

        header = tk.Frame(self.root, bg='#0a0a0a')
        header.pack(fill='x', padx=10, pady=5)
        tk.Label(header, text='PD2 INTEL TRADER', font=('Helvetica', 20, 'bold'),
                 fg='#00ff88', bg='#0a0a0a').pack(side='left')

        sf = tk.Frame(self.root, bg='#1a1a2e', relief='ridge', bd=1)
        sf.pack(fill='x', padx=10, pady=3)
        self.tick_label     = tk.Label(sf, text='Tick: --/300', font=('Courier', 11), fg='white', bg='#1a1a2e')
        self.pnl_label      = tk.Label(sf, text='PnL: $0.00',   font=('Courier', 11, 'bold'), fg='#00ff88', bg='#1a1a2e')
        self.position_label = tk.Label(sf, text='Pos: 0',       font=('Courier', 11), fg='white', bg='#1a1a2e')
        self.mode_label     = tk.Label(sf, text='Mode: --',     font=('Courier', 11), fg='yellow', bg='#1a1a2e')
        self.market_label   = tk.Label(sf, text='Mkt: -- / --', font=('Courier', 11), fg='white', bg='#1a1a2e')
        for w in [self.tick_label, self.pnl_label, self.position_label, self.mode_label]:
            w.pack(side='left', padx=10)
        self.market_label.pack(side='right', padx=10)

        ifl = tk.LabelFrame(self.root, text=' INTEL: Fair Value Range ',
                            font=('Helvetica', 12, 'bold'), fg='#00ccff', bg='#0a0a0a')
        ifl.pack(fill='x', padx=10, pady=5)
        fv_row = tk.Frame(ifl, bg='#0a0a0a')
        fv_row.pack(fill='x', padx=10, pady=3)
        self.fv_label         = tk.Label(fv_row, text='Fair Value: $50.00',      font=('Courier', 16, 'bold'), fg='#00ff88', bg='#0a0a0a')
        self.range_label      = tk.Label(fv_row, text='Range: [$40 - $60]',      font=('Courier', 12), fg='#cccccc', bg='#0a0a0a')
        self.confidence_label = tk.Label(fv_row, text='Confidence: 0%',          font=('Courier', 12, 'bold'), fg='red', bg='#0a0a0a')
        self.fv_label.pack(side='left')
        self.range_label.pack(side='left', padx=20)
        self.confidence_label.pack(side='right')
        bar_frame = tk.Frame(ifl, bg='#0a0a0a')
        bar_frame.pack(fill='x', padx=10, pady=2)
        self.confidence_bar = ttk.Progressbar(bar_frame, length=400, mode='determinate', maximum=100)
        self.confidence_bar.pack(fill='x')
        self.est_count_label = tk.Label(ifl, text='Estimates: 0 from 0/20 users',
                                        font=('Courier', 10), fg='#888888', bg='#0a0a0a')
        self.est_count_label.pack(anchor='w', padx=10, pady=2)

        tfl = tk.LabelFrame(self.root, text=' USER ESTIMATES ',
                            font=('Helvetica', 11, 'bold'), fg='#ffaa00', bg='#0a0a0a')
        tfl.pack(fill='both', expand=True, padx=10, pady=5)
        self.est_text = tk.Text(tfl, font=('Courier', 9), bg='#111122', fg='#cccccc',
                                height=15, wrap='none', selectbackground='#333355')
        sy = tk.Scrollbar(tfl, command=self.est_text.yview)
        sx = tk.Scrollbar(tfl, orient='horizontal', command=self.est_text.xview)
        self.est_text.configure(yscrollcommand=sy.set, xscrollcommand=sx.set)
        sy.pack(side='right', fill='y')
        sx.pack(side='bottom', fill='x')
        self.est_text.pack(fill='both', expand=True)

        bf = tk.Frame(self.root, bg='#0a0a0a')
        bf.pack(fill='x', padx=10, pady=5)
        tk.Button(bf, text='KILL ALL ORDERS', font=('Helvetica', 11, 'bold'),
                  bg='firebrick', fg='white', width=16, command=self.kill_all).pack(side='left', padx=5)
        tk.Button(bf, text='STOP BOT', font=('Helvetica', 11, 'bold'),
                  bg='gray25', fg='white', width=12, command=self.request_stop).pack(side='left', padx=5)
        self.alert_label = tk.Label(bf, text='READY', font=('Helvetica', 11, 'bold'),
                                    fg='#00ff88', bg='#0a0a0a')
        self.alert_label.pack(side='right', padx=10)

    def set_gateway(self, gw): self.gateway = gw
    def request_stop(self):
        self.stop_requested = True
        self.alert_label.config(text='STOP REQUESTED', fg='orange')
    def kill_all(self):
        if self.gateway:
            try:
                self.gateway.cancel_all_orders()
                self.alert_label.config(text='ORDERS KILLED', fg='red')
            except: pass

    def tick_ui(self):
        try:
            self.root.update_idletasks()
            self.root.update()
        except: pass

    def update(self, tick, pnl, inventory, mode, best_bid, best_ask, intel_summary, estimates_by_user):
        try:
            self.tick_label.config(text=f'Tick: {tick}/300')
            pc = '#00ff88' if pnl >= 0 else '#ff4444'
            self.pnl_label.config(text=f'PnL: ${pnl:.2f}', fg=pc)
            self.position_label.config(text=f'Pos: {inventory}')
            self.mode_label.config(text=f'Mode: {mode}')
            self.market_label.config(text=f'Mkt: {best_bid:.2f} / {best_ask:.2f}')
            fv   = intel_summary['fair_value']
            rl   = intel_summary['range_low']
            rh   = intel_summary['range_high']
            conf = intel_summary['confidence']
            self.fv_label.config(text=f'Fair Value: ${fv:.2f}')
            self.range_label.config(text=f'Range: [${rl:.2f} - ${rh:.2f}]  W=${intel_summary["range_width"]:.2f}')
            cp = int(conf * 100)
            cc = '#00ff88' if conf >= 0.8 else ('#ffaa00' if conf >= 0.5 else '#ff4444')
            self.confidence_label.config(text=f'Confidence: {cp}%', fg=cc)
            self.confidence_bar['value'] = cp
            self.est_count_label.config(
                text=f"Estimates: {intel_summary['total_estimates']} "
                     f"from {intel_summary['users_with_data']}/{intel_summary['active_users']} active users")
            self.est_text.config(state='normal')
            self.est_text.delete('1.0', 'end')
            self.est_text.insert('end', f"{'USER':<8} {'#':>3}  {'ESTIMATES':<45}  {'RANGE':>20}\n")
            self.est_text.insert('end', '-' * 80 + '\n')
            for user in ALL_USERS:
                ests = estimates_by_user.get(user, [])
                if not ests:
                    self.est_text.insert('end', f"{user:<8} {'0':>3}  {'-- no data --':<45}\n")
                else:
                    disp = ', '.join(f"t{e['tick']}:${e['estimate']:.2f}" for e in ests)
                    if len(disp) > 43: disp = disp[:40] + '...'
                    u_lo, u_hi = PRIOR_LOW, PRIOR_HIGH
                    for e in ests:
                        if e['tick'] is not None:
                            eb = (300 - e['tick']) / 50.0
                            u_lo = max(u_lo, e['estimate'] - eb)
                            u_hi = min(u_hi, e['estimate'] + eb)
                    self.est_text.insert('end', f"{user:<8} {len(ests):>3}  {disp:<45}  [${u_lo:.2f}-${u_hi:.2f}]\n")
            self.est_text.config(state='disabled')
            self.tick_ui()
        except: pass

In [32]:
def get_best_bid_ask(book):
    bids = book.get('bids', [])
    asks = book.get('asks', [])
    best_bid = float(bids[0]['price']) if bids else None
    best_ask = float(asks[0]['price']) if asks else None
    return best_bid, best_ask

def estimate_pnl(sec, mid):
    if sec is None: return 0.0
    if 'realized' in sec and 'unrealized' in sec:
        try: return float(sec['realized']) + float(sec['unrealized'])
        except: pass
    return float(sec.get('position', 0)) * mid

In [33]:
def run_pd2_trader(poll_delay=0.4):
    print('=' * 60)
    print('PD2 INTEL TRADER - Initializing...')
    print('=' * 60)
    print(f'[INIT] Remote hosts: {", ".join(REMOTE_HOSTS)}')
    print(f'[INIT] Local hosts: {", ".join(LOCAL_HOSTS)}')

    gateway, diagnostics = select_gateway(build_main_gateway_candidates(), timeout=4)
    if gateway is None:
        print('[INIT] No reachable RIT API gateway found.')
        for label, err in diagnostics:
            print(f'  - {label} -> {err}')
        print('[HINT] Current config expects BASE_URL=http://141.211.79.101:10002, API_PORT=10001, and the API key shown in RIT.')
        return pd.DataFrame()

    print(f'[INIT] Connected main gateway via {gateway.label}')
    intel = IntelAggregator(users=ALL_USERS, main_gateway=gateway, remote_hosts=REMOTE_HOSTS)
    trader = PD2Trader(intel=intel)
    ui = PD2Dashboard()
    ui.set_gateway(gateway)
    print(f'[INIT] {len(intel.users)} candidate users ready; active users will be auto-detected once the case is ACTIVE')

    loop_log = []
    trade_count = 0
    probed = False
    last_wait_state = None
    last_conn_error = None

    print('[INIT] Checking case status...')

    while True:
        try:
            if ui.stop_requested:
                print('[STOP] User stop.')
                try:
                    gateway.cancel_all_orders(timeout=1)
                except Exception:
                    pass
                break

            case_data = gateway.get_case(timeout=2)
            if case_data is None:
                conn_error = gateway.last_error or 'unknown error'
                if conn_error != last_conn_error:
                    print(f'[WAIT] Cannot reach selected gateway ({conn_error}), retrying...')
                    last_conn_error = conn_error
                ui.alert_label.config(text='Connection issue', fg='orange')
                ui.tick_ui()
                time.sleep(1)
                continue
            last_conn_error = None

            tick = case_data.get('tick', 0)
            status = str(case_data.get('status', '')).upper().strip()

            if status != 'ACTIVE':
                wait_state = (status, tick)
                if wait_state != last_wait_state:
                    print(f'[WAIT] status={status!r} tick={tick}')
                    last_wait_state = wait_state
                ui.alert_label.config(text=f'Status: {status}', fg='orange')
                ui.tick_ui()
                time.sleep(1)
                continue
            last_wait_state = None

            if not probed:
                print(f'[LIVE] Case is {status!r} at tick {tick}')
                intel.probe_users(timeout=1)
                probed = True

            if tick >= TOTAL_TIME:
                print(f'[END] tick={tick} status={status}')
                try:
                    gateway.cancel_all_orders(timeout=1)
                except Exception:
                    pass
                break

            new_est = intel.scrape_all_users(current_tick=tick)
            summary = intel.get_summary()
            if new_est > 0:
                print(f'[INTEL] +{new_est} | FV=${summary["fair_value"]:.2f} [{summary["range_low"]:.2f}-{summary["range_high"]:.2f}] conf={summary["confidence"]*100:.0f}%')

            book = gateway.get_book(TICKER, timeout=2)
            if book is None:
                time.sleep(poll_delay)
                continue
            best_bid, best_ask = get_best_bid_ask(book)
            if best_bid is None or best_ask is None:
                time.sleep(poll_delay)
                continue

            sec = gateway.get_security(TICKER, timeout=2)
            inventory = int(float(sec.get('position', 0))) if sec else 0
            mid_price = (best_bid + best_ask) / 2
            pnl = estimate_pnl(sec, mid_price)

            decision = trader.step(
                best_bid=best_bid,
                best_ask=best_ask,
                inventory=inventory,
                tick=tick,
                fair_value=summary['fair_value'],
                confidence=summary['confidence'],
                range_low=summary['range_low'],
                range_high=summary['range_high'],
            )
            mode = decision['mode']
            orders = decision.get('orders', [])

            if orders:
                gateway.cancel_all_orders(timeout=1)
                for order in orders:
                    action = order['action']
                    otype = order['type']
                    qty = int(order['quantity'])
                    price = order.get('price')
                    qty = min(qty, (POSITION_LIMIT - inventory) if action == 'BUY' else (POSITION_LIMIT + inventory))
                    qty = max(0, int(qty / 100) * 100)
                    if qty >= 100:
                        try:
                            if otype == 'MARKET':
                                gateway.submit_order(TICKER, 'MARKET', qty, action, timeout=1)
                            else:
                                gateway.submit_order(TICKER, 'LIMIT', qty, action, price=price, timeout=1)
                            trade_count += 1
                        except Exception as e:
                            print(f'[ORDER ERR] {e}')

            loop_log.append({
                'tick': tick,
                'best_bid': best_bid,
                'best_ask': best_ask,
                'mid': mid_price,
                'inventory': inventory,
                'pnl': pnl,
                'mode': mode,
                'confidence': summary['confidence'],
                'fair_value': summary['fair_value'],
                'range_low': summary['range_low'],
                'range_high': summary['range_high'],
                'total_estimates': summary['total_estimates'],
                'trade_count': trade_count,
            })

            ui.update(
                tick=tick,
                pnl=pnl,
                inventory=inventory,
                mode=mode,
                best_bid=best_bid,
                best_ask=best_ask,
                intel_summary=summary,
                estimates_by_user=intel.estimates_by_user,
            )

            print(f't={tick:>3} | inv={inventory:>6} | bid={best_bid:.2f} ask={best_ask:.2f} | FV=${summary["fair_value"]:.2f} [{summary["range_low"]:.2f}-{summary["range_high"]:.2f}] conf={summary["confidence"]*100:.0f}% | mode={mode} | pnl=${pnl:.2f}')
            time.sleep(poll_delay)

        except KeyboardInterrupt:
            print('\n[STOP] KeyboardInterrupt')
            try:
                gateway.cancel_all_orders(timeout=1)
            except Exception:
                pass
            break
        except Exception as e:
            print(f'[ERROR] {e}')
            time.sleep(1)

    df = pd.DataFrame(loop_log)
    if not df.empty:
        df.to_csv('pd2_run_log.csv', index=False)
        print(f'\nSaved pd2_run_log.csv ({len(df)} rows, {trade_count} trades)')
        print(f'Final PnL: ${df.iloc[-1]["pnl"]:.2f} | Inventory: {df.iloc[-1]["inventory"]} | FV: ${df.iloc[-1]["fair_value"]:.2f} | Conf: {df.iloc[-1]["confidence"]*100:.0f}%')
    return df

In [34]:
df = run_pd2_trader(poll_delay=0.4)

PD2 INTEL TRADER - Initializing...
[INIT] Remote hosts: http://141.211.79.101:10002, http://141.211.79.101:10001
[INIT] Local hosts: http://localhost:10001, http://localhost:10002, http://localhost:9999
[INIT] No reachable RIT API gateway found.
  - remote_basic:user4@http://141.211.79.101:10002 -> Timeout contacting http://141.211.79.101:10002/v1: HTTPConnectionPool(host='141.211.79.101', port=10002): Max retries exceeded with url: /v1/case (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x140b56540>, 'Connection to 141.211.79.101 timed out. (connect timeout=4)'))
  - remote_basic:user4@http://141.211.79.101:10001 -> ConnectionError contacting http://141.211.79.101:10001/v1: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))
  - local_api_key:http://localhost:10001 -> ConnectionError contacting http://localhost:10001/v1: HTTPConnectionPool(host='localhost', port=10001): Max retries exceeded with url: /v1/case (Caused by NewConnect

In [ ]:
if not df.empty:
    fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
    axes[0].plot(df['tick'], df['pnl'], 'g-', lw=1.5)
    axes[0].set_ylabel('PnL ($)')
    axes[0].set_title('PnL Over Time')
    axes[0].axhline(0, color='white', alpha=0.3)
    axes[0].grid(alpha=0.3)
    axes[1].fill_between(df['tick'], df['range_low'], df['range_high'], alpha=0.3, color='cyan', label='Intel Range')
    axes[1].plot(df['tick'], df['fair_value'], 'c-', lw=1.5, label='Fair Value')
    axes[1].plot(df['tick'], df['best_bid'], 'g--', lw=0.8, alpha=0.6, label='Bid')
    axes[1].plot(df['tick'], df['best_ask'], 'r--', lw=0.8, alpha=0.6, label='Ask')
    axes[1].set_ylabel('Price')
    axes[1].legend(loc='upper right', fontsize=8)
    axes[1].grid(alpha=0.3)
    axes[2].plot(df['tick'], df['inventory'], 'y-', lw=1.2)
    axes[2].set_ylabel('Inventory')
    axes[2].set_xlabel('Tick')
    axes[2].axhline(0, color='white', alpha=0.3)
    axes[2].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()